# ECMWF Web API exercises — Answers

In [ ]:
from ecmwfapi import ECMWFService
import earthkit.data

### 1) Answer

The field is retrieved in its native **spherical harmonic representation** (visible in the MARS request log as `REPRES = SH`). NetCDF can only store data on a **regular lat/lon grid or a regular Gaussian grid**, so the conversion step (`grib_to_netcdf`, run internally by the Web API) rejects it:

```
ECCODES ERROR   :  Grid type = sh
ECCODES ERROR   :  First GRIB is not on a regular lat/lon grid or on a regular Gaussian grid. Exiting.
```

The fix is to add a `grid` keyword (e.g. `"grid": "1/1"`) to the request. This makes MARS **interpolate** the field onto a regular grid *before* the NetCDF conversion step runs, so the conversion then succeeds — as shown below.

In [ ]:
server = ECMWFService("mars")

request = {
    "class": "od",
    "date": "-1",
    "expver": "1",
    "param": "130.128",  # temperature
    "levtype": "pl",
    "levelist": "500",
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "step": "0",
    "grid": "1/1",  # <-- interpolate onto a regular grid first
    "format": "netcdf",
}
server.execute(request, "../data/ex1_fixed.nc")

## Exercise 2: how many tapes?

MARS can estimate the **cost** of a request — number of fields, disk files, tape files, total size — *without* actually retrieving any data, using the `list,output=cost,...` action.

In [ ]:
server = ECMWFService("mars")
server.execute(
    "list,"
    "output=cost,"
    "class=od,"
    "stream=oper,"
    "expver=1,"
    "type=fc,"
    "levtype=sfc,"
    "param=165.128/166.128/167.128,"
    "date=-3,"
    "time=12,"
    "step=0/to/120/by/6",
    "../data/cost_exercise.txt"
)

In [ ]:
with open("../data/cost_exercise.txt") as f:
    print(f.read())

0 Tapes and 6 disk files

### Answer

From the printed output above, `number_of_tapes=0` and `number_of_disk_files=6` — all 63 fields for this request are **online** (on disk), so MARS does not need to access any tape at all.

In general, older or less frequently accessed data may be stored **offline on tape**; `number_of_tapes` would then be greater than 0, and such requests are much slower (and effectively more "expensive") to retrieve, since a physical tape needs to be mounted and read. Running `list,output=cost,...` first — like we just did — lets you check this *before* committing to a full retrieval.

## Exercise 3: multi-target request

MARS supports a *multi-target* syntax, where the `target` contains a bracketed keyword, e.g. `target="name_[step].grib"`, so that one output file is automatically created per unique value of that keyword — but this only works when running the MARS client directly on ECMWF's own compute.

Below we write a request for 2m temperature at steps `0/6/12`, using the bracket syntax above as the `target` through the Web API.

In [ ]:
server = ECMWFService("mars")

request = {
    "class": "od",
    "date": "-1",
    "expver": "1",
    "param": "167.128",  # 2m temperature
    "levtype": "sfc",
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "step": "0/6/12",
    "grid": "1/1",
}
server.execute(request, "../data/ex3_multi_[step].grib")

### Answer

**Only 1 file is generated** — literally named `ex3_multi_[step].grib` (brackets and all!). The Web API does not implement MARS's server-side multi-target splitting feature; that is only available when running the MARS client directly on ECMWF's own compute.

To get the same result through the Web API, retrieve everything into a single file and then split it **client-side**, e.g. with `earthkit.data`'s `to_target("file-pattern", ...)` (demonstrated in the examples notebook) or with command-line tools like `grib_copy`/`grib_filter`.